# CLM-Workspace-DepthKV-001

T4 ×2、Internet、GITHUB_TOKEN；Save & Run All。
复用数据/Trainer/发布器：工程门禁 → Seed17四组Smoke → 实测预算 → C2/C-DKV/T0-P/T0-F × Seed18001/18002/18003 各2M → 验证和真实推理测速。
FP32，单次120分钟；不减Targets，已完成模型不重训，科学结论等待研究复核。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/clm-workspace-depth-kv-001'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-workspace-depth-kv-001'
OUTPUT = WORKSPACE / 'clm-depth-kv-bootstrap'
HARD_LIMIT_SECONDS = 7200
started = time.monotonic()
PLATFORM_RESERVE_SECONDS = 60
deadline = started + HARD_LIMIT_SECONDS - PLATFORM_RESERVE_SECONDS

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; partial reports and final models, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('clm-depth-kv-' + SOURCE_COMMIT[:12])

import importlib.util
missing = [name for name in ('datasets','tokenizers','huggingface_hub','pytest') if importlib.util.find_spec(name) is None]
if missing:
    run_bounded([sys.executable, '-m', 'pip', 'install', '--quiet'] + missing, timeout_seconds=600)
run_bounded([sys.executable, str(REPO / 'research/projects/cellular-language-models/experiments/clm-workspace-depth-kv-001/execute.py'),
    '--output', str(OUTPUT), '--started', str(started)], cwd=REPO)
